# Notebook de Organização

Notebook (1/5) da Tarefa 2 — Projeto CineData Analytics

Este notebook prepara a estrutura do Unity Catalog para a arquitetura medalhão. Ele cria o catálogo `cinedata_medallion`, os schemas das três camadas (`bronze`, `silver` e `gold`) e o volume `landing`, onde ficam os arquivos CSV de origem.

## Decisões adotadas
- **Schema = database:** no Databricks, `CREATE SCHEMA` e `CREATE DATABASE` são equivalentes. Cada camada da arquitetura é um schema dentro do mesmo catálogo.
- **Volume dentro do Bronze:** o volume `landing` pertence ao schema `bronze`, porque os arquivos crus são a origem dessa camada. No Unity Catalog, arquivos só podem existir dentro de um volume, e o caminho segue o padrão `/Volumes/<catálogo>/<schema>/<volume>`.
- **Execução idempotente:** todos os comandos usam `IF NOT EXISTS`, então rodar o notebook mais de uma vez não gera erro nem recria objetos.
- **Nomes em minúsculas:** o Unity Catalog armazena os nomes de objetos em minúsculas, por isso os caminhos sempre usam minúsculas.

In [0]:
#parametros do ambiente

catalog ="cinedata_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

# criação dos nomes/caminhos dos schemas de cada camada
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# criação do nome/caminho volume para csvs
landing_path = f"/Volumes/{catalog}/{bronze_schema_name}"

# verificação da configuração
print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")



In [0]:

# criação dos dos schemas de cada camada no banco
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

display(spark.sql(f"SHOW SCHEMAS IN {catalog}"))

# criação do volume para csvs
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

display(spark.sql(f"SHOW VOLUMES IN {bronze_schema}"))